# Chapter 14 — Exercise Solutions## Reward Modeling[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 14.1: Preference Pair Generation from Reward Model

In [ ]:
import numpy as np, torch

# Simulate reward model scoring (since we don't have a real trained RM)
def simulated_reward_model(article: str, summary: str) -> float:
    """Simulated RM score based on heuristics (for demonstration)."""
    score = 0.0
    # Length heuristic: summaries between 30-80 words score higher
    words = len(summary.split())
    if 30 <= words <= 80: score += 0.3
    # Coverage: mentions key words from article
    article_words = set(article.lower().split())
    summary_words = set(summary.lower().split())
    overlap = len(article_words & summary_words) / max(len(article_words), 1)
    score += overlap * 0.5
    # Fluency proxy: no repeated consecutive words
    tokens = summary.lower().split()
    repeats = sum(1 for i in range(len(tokens)-1) if tokens[i]==tokens[i+1])
    score -= repeats * 0.2
    return float(np.clip(score, 0, 1))

def generate_preference_pairs(articles, base_model_fn, reward_model_fn,
                               n_samples=4, margin_threshold=0.2):
    """
    Generate DPO preference pairs:
    1. Sample N responses per article from base model
    2. Score each with reward model
    3. Pair highest-scoring (chosen) with lowest-scoring (rejected)
    4. Keep pairs with reward gap > margin_threshold
    """
    pairs = []
    for article in articles:
        # Sample N summaries (simulated here)
        summaries = [base_model_fn(article, seed=i) for i in range(n_samples)]
        scores    = [reward_model_fn(article, s) for s in summaries]

        chosen_idx   = np.argmax(scores)
        rejected_idx = np.argmin(scores)
        reward_gap   = scores[chosen_idx] - scores[rejected_idx]

        if reward_gap > margin_threshold:
            pairs.append({
                'prompt':   article,
                'chosen':   summaries[chosen_idx],
                'rejected': summaries[rejected_idx],
                'chosen_reward':   scores[chosen_idx],
                'rejected_reward': scores[rejected_idx],
                'reward_gap':      reward_gap,
            })

    print(f"Generated {len(pairs)}/{len(articles)} pairs with gap > {margin_threshold}")
    return pairs

# Simulated base model (generates varied summaries)
SUMMARIES = [
    "Scientists created fast-charging batteries reaching 80% in 5 minutes with new anode material.",
    "Battery technology advances with MIT research team working on next-generation solutions.",
    "battery battery battery fast charge fast charge fast",  # bad (repetitive)
    "MIT scientists have invented a revolutionary battery charging breakthrough using novel anode.",
]

def sim_base_model(article, seed=0):
    return SUMMARIES[seed % len(SUMMARIES)]

article = "Scientists at MIT developed a battery that charges to 80% in 5 minutes using a novel anode."
pairs = generate_preference_pairs([article]*5, sim_base_model, simulated_reward_model)

print("\nSample preference pair:")
if pairs:
    p = pairs[0]
    print(f"  Chosen  (reward={p['chosen_reward']:.3f}):   {p['chosen'][:80]}")
    print(f"  Rejected (reward={p['rejected_reward']:.3f}): {p['rejected'][:80]}")
    print(f"  Reward gap: {p['reward_gap']:.3f}")
    print(f"  Fraction with gap > 0.3: {sum(1 for p in pairs if p['reward_gap']>0.3)/len(pairs):.0%}")

# YOUR TURN: With 4 samples per prompt, what fraction of pairs have gap > 0.3?
# Try n_samples = 2, 4, 8, 16 — how does pair quality scale with n_samples?


### Solution 14.2: Reward Hacking Demo

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def naive_reward_model(response: str) -> float:
    """A naive reward model that can be gamed."""
    score = 0.0
    # Rewards longer responses (naive length bias)
    score += min(len(response.split()) / 100, 0.4)
    # Rewards certain keywords (can be gamed)
    keywords = ['certainly', 'absolutely', 'great question', 'excellent', 'happy to help']
    score += sum(0.1 for k in keywords if k.lower() in response.lower())
    # Slight grammar bonus (simplistic)
    score += 0.1 if response[0].isupper() else 0
    return min(score, 1.0)

def honest_reward_model(response: str, ground_truth: str) -> float:
    """A better reward model that checks actual quality."""
    # Semantic overlap with ground truth (simplified)
    resp_words  = set(response.lower().split())
    truth_words = set(ground_truth.lower().split())
    overlap = len(resp_words & truth_words) / max(len(truth_words), 1)
    return min(overlap + 0.1, 1.0)

# Simulate reward hacking: optimising naive RM produces bad outputs
prompt = "What is the capital of France?"
ground_truth = "Paris is the capital of France."

honest_responses = [
    "Paris is the capital of France.",
    "France's capital city is Paris.",
    "The capital is Paris, located in northern France.",
]

hacked_responses = [
    "Certainly! That's an absolutely excellent question. I'm absolutely happy to help you with this great question. Paris " + "is the capital " * 20,
    "Great question! Absolutely certainly the capital of France is Paris. Excellent inquiry! Happy to help! " * 3,
    "Certainly absolutely certainly absolutely great question excellent happy to help Paris capital France certainly absolutely",
]

print("Naive RM vs Honest RM comparison:")
print(f"{'Response Type':<20} {'Naive RM':>10} {'Honest RM':>10}")
print("-" * 44)
for r in honest_responses:
    print(f"{'Honest':<20} {naive_reward_model(r):>10.3f} {honest_reward_model(r, ground_truth):>10.3f}")
for r in hacked_responses:
    print(f"{'Hacked':<20} {naive_reward_model(r):>10.3f} {honest_reward_model(r, ground_truth):>10.3f}")

print("\nKey insight: hacked responses score HIGH on naive RM but LOW on honest RM")
print("This is reward hacking — the policy maximises the proxy, not the true objective")
print("Prevention: test RM on adversarial examples BEFORE using it for training")

# YOUR TURN: Design a reward function that is ROBUST to the gamed responses above
# Hint: penalise repetition, cap length bonus, measure semantic density
